# Hausverwaltungs-Chatbot mit Langfuse-Tracing

Ein Telefon-Agent für eine Hausverwaltung, der in drei LLM-Schritten arbeitet —
und dabei jeden Schritt als **Trace** an Langfuse meldet, damit man im Dashboard
nachvollziehen kann, was das Modell wann entschieden hat.

1. **Begrüßung** (`step-greeting`) — freundlich antworten und den Namen mitnehmen,
   falls der Mieter ihn schon genannt hat; sonst danach fragen
2. **Verifizierung** (`step-auth`) — Name + Adresse prüfen, strukturierte Antwort
3. **Routing** (`step-routing`) — Anliegen einer von fünf Abteilungen zuordnen

Alle drei Schritte eines Anrufs landen unter **einem gemeinsamen Trace**.

## Datenfluss — und was dabei protokolliert wird

Wichtig für das Verständnis: Die LLM-Aufrufe sind **zustandslos** — kein Schritt
bekommt den Antworttext eines vorherigen Schritts als Kontext. Weitergereicht wird
genau ein extrahierter Wert: der Name, den Schritt 1 aus der Eröffnung zieht, falls
er dort schon gefallen ist. Alles andere sind rohe Nutzereingaben.

Links steht, was im Programm passiert, rechts, was Langfuse davon aufzeichnet:

```
    opening (Eröffnung des Mieters, evtl. schon mit Namen)      LANGFUSE
            |
            v
    +---------------------+  LLM-Input: opening
    | 1. step-greeting    |----> GreetingResult                 Span  step-greeting
    +---------------------+      reply: Text an den Mieter       └─ OpenAI-Generation
            |                    name: aus der Eröffnung
            |                          oder None
      name gefunden? --nein--> input("Mieter (Name): ")         Event name-requested
            | ja
            v
    name, address
            v
    +---------------------+  LLM-Input: Name + Adresse
    | 2. step-auth        |----> AuthResult                     Span  step-auth
    +---------------------+      verified (Abbruch?)             ├─ Span address-format-check
            |                    customer_name (Anrede)          └─ OpenAI-Generation
            |                                                   Event auth-result
      verified? --nein--> Ende                                  Event call-aborted
            | ja                                                Tag   auth-failed
            v
    issue (konkretes Anliegen)
            v
    +---------------------+  LLM-Input: transcript =
    | 3. step-routing     |  opening + name + address + issue   Span  step-routing
    +---------------------+----> RoutingDecision                 ├─ OpenAI-Generation
            |                    confidence (Selbsteinschätzung)  └─ Score routing-confidence
            v
    Ausgabe an den Mieter                                       Event department-handoff
                                                                Tags + Metadata am Trace
                                                                Score routing-confidence
                                                                Score routing-department
```

## Der Trace im Dashboard

Aus einem Anruf wird genau ein Trace. So sieht sein Baum aus — die Einrückung
entsteht dadurch, dass jede Observation ein Kind der gerade aktiven wird:

```
tenant-routing-call                 @observe        Wurzel; session_id, tags, metadata
│                                                   kommen über propagate_attributes
├── step-greeting                   @observe
│   └── OpenAI-Generation           langfuse.openai  Prompt, Antwort, Tokens, Kosten
├── name-requested                  create_event     nur wenn nachgefragt werden musste
├── step-auth                       @observe
│   ├── address-format-check        start_as_current_observation + span.update
│   └── OpenAI-Generation           langfuse.openai
├── auth-result                     create_event     level=WARNING bei verified=False
├── call-aborted                    create_event     nur im Abbruchpfad — danach Ende
├── step-routing                    @observe         + Score routing-confidence
│   └── OpenAI-Generation           langfuse.openai
└── department-handoff              create_event     Meilenstein "Ticket ist raus"
```

Scores sind keine Observationen und tauchen deshalb nicht als Knoten im Baum auf:
Sie hängen als benannte Werte an einer Observation (`step-routing`) oder am Trace
selbst — dort werden `routing-confidence` und `routing-department` zu eigenen
Spalten in der Trace-Liste. `routing-confidence` trägt dank einer **Score-Config**
beides zugleich: das Wort (`"hoch"`) und die Zahl dahinter (`1.0`).

`call-aborted` und alles darunter schließen sich gegenseitig aus: Entweder der
Anruf bricht bei der Verifizierung ab, oder er läuft bis zur Übergabe durch.

Welcher Aufruf im Code was davon erzeugt:

| Aufruf | was im Dashboard daraus wird |
| --- | --- |
| `from langfuse.openai import openai` | pro API-Aufruf eine **Generation**: Prompt, Antwort, Modell, Token-Zahlen, geschätzte Kosten, Dauer |
| `@observe(name=...)` | ein **Span** um den Funktionsaufruf; Argumente werden Input, Rückgabewert Output, Exceptions Fehlerstatus |
| `langfuse.start_as_current_observation(...)` | ein **Span** von Hand — für Logik ohne LLM; `span.update(metadata=...)` reichert ihn an |
| `langfuse.create_event(...)` | ein **Event**: ein Zeitpunkt ohne Dauer |
| `langfuse.score_current_span(...)` | ein **Score** an der aktiven Observation: Name, Wert, Datentyp — im Dashboard aggregierbar |
| `langfuse.score_current_trace(...)` | derselbe Score, aber am Trace — eigene Spalte in der Trace-Liste |
| `langfuse.api.score_configs.create(...)` | eine **Score-Config**: einmalige Projekt-Einstellung, die zu jedem Kategorie-Wort die Zahl festlegt |
| `propagate_attributes(...)` | `session_id`, `tags`, `metadata` am aktiven Span und an allen, die im Block noch entstehen |
| `langfuse.flush()` | schickt den Puffer sofort an die API |


## Setup: Pakete importieren

Der Import `from langfuse.openai import openai` ist der zentrale Trick: Dieser
Wrapper verhält sich exakt wie das normale `openai`-Paket, fängt aber jeden
Aufruf ab und schickt Prompt, Antwort, Modell, Token-Zahl und Dauer automatisch
an Langfuse. Am eigentlichen API-Code ändert sich dadurch **nichts**.

In [95]:
import os
import uuid

from dotenv import load_dotenv
from pydantic import BaseModel

# Drop-in-Ersatz fuer "import openai": identische API, aber jeder Aufruf wird
# automatisch als Generation-Observation aufgezeichnet - Prompt, Antwort, Modell,
# Token-Zahlen, geschaetzte Kosten in USD, Dauer und eventuelle Fehler. Am
# eigentlichen API-Code aendert sich dadurch keine einzige Zeile.
from langfuse.openai import openai

# Die drei Bausteine, die in diesem Notebook vorkommen:
#   get_client()          Client-Objekt, gebaut aus den LANGFUSE_*-Umgebungs-
#                         variablen. Singleton: jeder Aufruf liefert dasselbe
#                         Objekt. Noetig fuer auth_check(), fuer manuelle Spans
#                         und fuer flush().
#   observe               Dekorator - macht aus einem Funktionsaufruf eine
#                         Observation (einen Span) im Trace.
#   propagate_attributes  Context Manager - haengt session_id, tags und metadata
#                         an den gerade aktiven Span und an alle, die innerhalb
#                         des Blocks noch entstehen.
from langfuse import get_client, observe, propagate_attributes


## Konfiguration

Aus der `.env`-Datei kommen zwei Dinge: der OpenAI-Schlüssel (`OPENAI_API_KEY`)
und die Langfuse-Zugangsdaten (`LANGFUSE_PUBLIC_KEY`, `LANGFUSE_SECRET_KEY`,
`LANGFUSE_HOST`). `get_client()` liest sie selbst aus der Umgebung — man muss
nichts übergeben.

`auth_check()` sagt sofort, ob die Verbindung zu Langfuse steht. Ohne diesen
Test merkt man einen Tippfehler im Schlüssel erst daran, dass das Dashboard
leer bleibt.

In [96]:
load_dotenv()

model = os.getenv("LLM_MODEL", "gpt-4o-mini")

# Liest LANGFUSE_PUBLIC_KEY, LANGFUSE_SECRET_KEY und LANGFUSE_HOST selbst aus der
# Umgebung - es wird nichts uebergeben. Der Client puffert Traces im Hintergrund
# und schickt sie gebuendelt los (siehe flush() am Ende des Notebooks).
langfuse = get_client()

print(f"Modell: {model}")
# auth_check() fragt synchron die Projekt-API ab und sagt, ob die Keys gueltig
# sind. Es legt keinen Trace an. Nur fuer solche Checks gedacht - in Produktiv-
# code hat der Aufruf nichts verloren, weil er blockiert.
print(f"Langfuse-Verbindung ok: {langfuse.auth_check()}")


Modell: gpt-4.1-mini
Langfuse-Verbindung ok: True


## Die Abteilungen und die Datenmodelle

`DEPARTMENTS` ist die Wissensbasis des Routings — die Schlüssel landen später
wörtlich im System-Prompt, damit das Modell weiß, wohin es überhaupt verteilen darf.

Die drei Pydantic-Klassen — eine pro Schritt — sind mehr als Typ-Deklarationen:
Sie werden gleich als `response_format` an die API übergeben. Das Modell muss dann
JSON liefern, das exakt zu diesen Feldern passt — kein Parsen von Freitext, keine
Überraschungen. Bei den Feldern lohnt der genaue Blick, denn sie steuern den
weiteren Programmablauf: `verified` entscheidet über den Abbruch, `department`
über das Ziel.


In [68]:
# ---------------------------------------------------------------------------
# DEPARTMENTS: die Auswahlmenge fuer das Routing
#
#   Schluessel -> stabile, maschinenlesbare Abteilungs-ID
#   Wert       -> Beschreibung in natuerlicher Sprache
#
# Die Beschreibungen sind kein Kommentar fuer Menschen, sondern Teil des Prompts:
# route_to_department() (Schritt 3) rendert das Dictionary zu einer Aufzaehlung
# und haengt sie in den System-Prompt. Das Modell darf danach ausschliesslich
# einen dieser Schluessel zurueckgeben -> RoutingDecision.department.
#
# Daraus folgt fuer die Praxis:
#   * Neue Abteilung = ein Eintrag mehr. Prompt, Auswahlmenge und damit das
#     Verhalten des Agenten ziehen automatisch nach - kein Code-Update noetig.
#   * Die Schluessel wandern spaeter ins Ticketsystem und in die Langfuse-Filter.
#     Also: Beschreibungen nachschaerfen ja, Schluessel umbenennen nur bewusst.
#   * Ueberschneidende Beschreibungen sind die haeufigste Ursache fuer unsicheres
#     Routing (erkennbar an RoutingDecision.confidence == "niedrig").
# ---------------------------------------------------------------------------
DEPARTMENTS = {
    "rental-contracts":     "Mietverträge — Fragen zum Mietvertrag, Verlängerungen, Änderungen",
    "terminations-moveout": "Kündigungen & Auszug — Kündigungen, Auszugstermine, Kautionsrückzahlung",
    "tenant-complaints":    "Mieterbeschwerden — Lärm, Nachbarschaftsstreit, allgemeine Beschwerden",
    "energy-heating":       "Energie & Heizung — Heizungsausfälle, Warmwasser, Nebenkostenabrechnung",
    "repairs-maintenance":  "Reparaturen & Instandhaltung — defekte Einrichtungen, Gebäudeschäden, allgemeine Reparaturen",
}


In [69]:
class GreetingResult(BaseModel):
    """Ergebnis der Begruessung (Schritt 1).

    Der Text an den Mieter und - falls vorhanden - der Name, den er in der
    Eroeffnung schon genannt hat.
    """

    reply: str         # was der Agent dem Mieter antwortet
    name: str | None   # Name aus der Eroeffnung, sonst None. Das Optional ist
                       # Absicht: None ist die ehrliche Antwort, wenn kein Name
                       # gefallen ist - ohne dieses Ventil erfindet das Modell einen.


class AuthResult(BaseModel):
    """Ergebnis der Mieter-Verifizierung (Schritt 2, verify_tenant).

    Wird als `response_format=AuthResult` an die API uebergeben. Das Modell muss
    dann JSON mit exakt diesen drei Feldern liefern; die Antwort steckt fertig
    geparst in `response.choices[0].message.parsed` - also ein echtes Objekt mit
    Attributzugriff, kein Freitext, den man selbst zerlegen muesste.
    """

    verified: bool         # Steuert den Programmfluss: False -> handle_call bricht
                           # ab, es wird gar nicht erst geroutet (Tag "auth-failed").
    customer_name: str     # Name so, wie das Modell ihn aus der Eingabe gelesen hat.
    reason: str            # Begruendung im Klartext - landet im Langfuse-Trace und
                           # macht spaeter nachvollziehbar, WARUM abgelehnt wurde.


class RoutingDecision(BaseModel):
    """Ergebnis des Routings (Schritt 3, route_to_department).

    Gleiches Prinzip wie oben, zusaetzlich der Rueckgabewert des gesamten
    Anrufs: was handle_call() am Ende liefert und was ein Ticketsystem
    weiterverarbeiten wuerde.
    """

    department: str        # Genau einer der fuenf DEPARTMENTS-Schluessel oben.
                           # Das Zielfeld: es entscheidet, wer den Anruf bekommt.
    routing_reason: str    # Warum diese Abteilung - wichtig fuer spaetere Analysen
                           # ("warum landen so viele Faelle bei energy-heating?").
    issue_summary: str     # Kurzfassung des Anliegens fuer die Uebergabe, damit die
                           # Abteilung nicht das ganze Transkript lesen muss.
    confidence: str        # "niedrig" / "mittel" / "hoch" - Selbsteinschaetzung des
                           # Modells, kein gemessener Wert. Niedrig = Kandidat fuer
                           # manuelle Pruefung; in Langfuse gut filterbar.


## Schritt 1: Begrüßung

Der erste Schritt erledigt zwei Dinge auf einmal: freundlich antworten **und**
den Namen mitnehmen, falls der Mieter ihn schon genannt hat („Guten Tag, hier ist
Anna Schmidt …"). Deshalb kommt hier kein freier Text zurück, sondern ein
`GreetingResult` mit den Feldern `reply` und `name`.

**Strukturierte Ausgabe:** Statt `chat.completions.create` wird
`beta.chat.completions.parse` mit `response_format=GreetingResult` aufgerufen.
Die Antwort steckt dann nicht in `.content`, sondern fertig geparst in `.parsed` —
ein echtes Objekt mit Attributzugriff, kein Freitext, den man selbst zerlegen
müsste. Dass `name` auch `None` sein darf, ist dabei kein Detail: Ohne dieses
Ventil müsste das Modell das Feld füllen und würde einen Namen erfinden.

**Tracing:** Der Dekorator `@observe(name="step-greeting")` macht aus dem
Funktionsaufruf einen **Span** im Trace: Langfuse misst die Dauer, merkt sich
Argumente und Rückgabewert und hängt den darin stattfindenden OpenAI-Aufruf als
Kind darunter. Im Dashboard sieht man später also die Verschachtelung
`step-greeting` → OpenAI-Call.

Ob überhaupt nachgefragt werden muss, entscheidet danach kein LLM mehr, sondern
eine Zeile Python: `name = greeting.name or nachfragen()`.


In [ ]:
# @observe() legt um den gesamten Funktionsaufruf eine Observation vom Typ "span":
# Langfuse misst Start und Ende, schreibt die Argumente als Input und den
# Rueckgabewert als Output in den Span (hier also das ganze GreetingResult) und
# vermerkt eine Exception als Fehlerstatus. Ohne name=... hiesse der Span wie die
# Funktion.
#
# Der OpenAI-Aufruf im Rumpf erzeugt durch den langfuse.openai-Wrapper eine
# eigene Generation-Observation und haengt sich automatisch als Kind darunter:
#     step-greeting
#        - OpenAI-Generation (Prompt, Antwort, Tokens, Kosten, Dauer)
#
# Wer die Funktion aufruft, entscheidet ueber die Einordnung: aus einer anderen
# @observe-Funktion heraus wird sie deren Kind, direkt aufgerufen ist sie selbst
# die Wurzel eines eigenen Traces.
@observe(name="step-greeting")
def greet_and_collect_name(customer_message: str) -> GreetingResult:
    response = openai.beta.chat.completions.parse(
        model=model,
        messages=[
            {"role": "system", "content": (
                "Du bist eine freundliche Empfangskraft bei der Hausverwaltung. "
                "Begrüße den Mieter herzlich und antworte immer auf Deutsch. "
                "Trage in 'name' den Namen des Mieters ein, falls er ihn in seiner "
                "Nachricht genannt hat, sonst null. Erfinde niemals einen Namen. "
                "Nur wenn 'name' null ist, frage in 'reply' nach dem Namen."
            )},
            {"role": "user", "content": customer_message}
        ],
        response_format=GreetingResult
    )
    return response.choices[0].message.parsed


In [71]:
# Beide Faelle nebeneinander: Eroeffnung mit Namen und ohne.
# Erwartung: oben ein Name und keine Rueckfrage, unten None und eine Rueckfrage.
opening = "Guten Tag, bei mir in der Wohnung ist die Heizung ausgefallen."

for text in ["Guten Tag, hier ist Anna Schmidt, bei mir ist die Heizung ausgefallen.", opening]:
    greeting = greet_and_collect_name(text)
    print("name: ", repr(greeting.name))
    print("reply:", greeting.reply)
    print("-" * 70)


name:  'Anna Schmidt'
reply: Guten Tag Frau Schmidt, herzlich willkommen! Es tut mir leid zu hören, dass Ihre Heizung ausgefallen ist. Wir kümmern uns darum so schnell wie möglich.
----------------------------------------------------------------------
name:  None
reply: Guten Tag! Es tut mir leid zu hören, dass Ihre Heizung ausgefallen ist. Damit wir Ihnen schnell weiterhelfen können, darf ich bitte Ihren Namen erfahren?
----------------------------------------------------------------------


## Schritt 2: Verifizierung

`AuthResult` als `response_format` folgt demselben Muster wie in Schritt 1: Das
Modell muss JSON liefern, das exakt zu den Feldern passt, und die geparste Antwort
steht in `.parsed`. Das Feld `verified` entscheidet über Abbruch oder Weitermachen
— als `bool` lässt es sich direkt in ein `if` schreiben.

Dazu kommt ein **manueller Sub-Span**: Der Block
`with langfuse.start_as_current_observation(...)` erzeugt eine eigene Beobachtung
*innerhalb* der Funktion. Das ist nützlich für Logik, die gar kein LLM benutzt —
hier eine simple Plausibilitätsprüfung der Adresse. Über `span.update(metadata=...)`
landen beliebige Werte im Dashboard, nach denen man später filtern kann.

Hinweis: `is_plausible` wird bewusst nur protokolliert, nicht als Abbruchkriterium
verwendet — die Entscheidung trifft allein das Modell.


In [72]:
@observe(name="step-auth")
def verify_tenant(name: str, address: str) -> AuthResult:
    # start_as_current_observation() erzeugt eine Observation von Hand, statt sie
    # an eine Funktion zu haengen - praktisch fuer Logik ganz ohne LLM. "as_current"
    # heisst: Der Span wird zum aktiven Kontext, alles darin Erzeugte haengt sich
    # als Kind darunter. Am Ende des with-Blocks wird er automatisch beendet und
    # seine Dauer steht fest. Ohne as_type=... ist es ein normaler "span"; moeglich
    # waeren u. a. "tool", "retriever" oder "generation".
    with langfuse.start_as_current_observation(name="address-format-check") as span:
        is_plausible = len(address.split()) >= 2
        # span.update() schreibt Felder in genau diese eine Observation (nicht in
        # den Trace). metadata sind freie Schluessel-Wert-Paare; die Werte werden
        # zu Strings gemacht und im Dashboard filter- und durchsuchbar.
        span.update(metadata={"raw_address": address, "passed_format": is_plausible})

    response = openai.beta.chat.completions.parse(
        model=model,
        messages=[
            {"role": "system", "content": (
                "Du simulierst ein Mieter-Verifizierungssystem. "
                "Wenn die Adresse plausibel klingt (Straßenname + Hausnummer + Stadt), "
                "markiere den Mieter als verifiziert. "
                "Schreibe die Begründung (reason) auf Deutsch."
            )},
            {"role": "user", "content": f"Name: {name}\nAdresse: {address}"}
        ],
        response_format=AuthResult
    )
    return response.choices[0].message.parsed

In [58]:
name = "Anna Schmidt"
address = "Hauptstraße 12, 10115 Berlin"

auth = verify_tenant(name, address)
print(auth)                 # ein AuthResult-Objekt, kein Text
print()
print("verified:     ", auth.verified)
print("customer_name:", auth.customer_name)
print("reason:       ", auth.reason)

verified=True customer_name='Anna Schmidt' reason="Die Adresse 'Hauptstraße 12, 10115 Berlin' ist plausibel, da sie einen gültigen Straßennamen, eine Hausnummer und eine bekannte Stadt enthält."

verified:      True
customer_name: Anna Schmidt
reason:        Die Adresse 'Hauptstraße 12, 10115 Berlin' ist plausibel, da sie einen gültigen Straßennamen, eine Hausnummer und eine bekannte Stadt enthält.


In [73]:
# Gegentest: eine unbrauchbare Adresse. Das Modell sollte verified=False liefern.
print(verify_tenant("Anna Schmidt", "keine Ahnung"))

verified=False customer_name='Anna Schmidt' reason="Die angegebene Adresse 'keine Ahnung' ist nicht plausibel, da sie keinen gültigen Straßenname, Hausnummer und Stadt enthält."


## Schritt 3: Routing

Der System-Prompt wird hier **dynamisch** zusammengebaut: `dept_list` rendert das
`DEPARTMENTS`-Dictionary als Aufzählung in den Prompt hinein. Neue Abteilung
anlegen heißt damit nur, den Dictionary-Eintrag zu ergänzen — der Prompt zieht
automatisch nach.

Als User-Message geht das gesamte Transkript rein. Das ist die einzige Stelle,
an der die vorher gesammelten Informationen zusammenlaufen: `opening`, `name`,
`address` und `issue` als ein String. Zurück kommt wieder ein geparstes
Pydantic-Objekt.

## Scores: das Ergebnis bewerten

Bis hierhin protokolliert das Notebook, **was** passiert ist — Spans, Events,
Tags, Metadata. Ein **Score** beantwortet die nächste Frage: *wie gut* war es.

Der Unterschied zu Metadata ist kein kosmetischer. Metadata ist ein freier
Schlüssel-Wert-Anhang, nach dem man filtern kann. Ein Score ist ein eigenes
Objekt mit Name, Wert und Datentyp — Langfuse kann darüber mitteln, ihn über die
Zeit auftragen, Traces danach sortieren und zwei Prompt-Versionen daran
vergleichen. Deshalb wandert `confidence` gleich in beides: als Tag zum Filtern
und als Score zum Auswerten.

### Wort oder Zahl? Beides.

`confidence` ist ein Wort (`"niedrig"`/`"mittel"`/`"hoch"`), aggregieren lässt
sich aber nur eine Zahl. Naheliegend wäre, das Wort im Code zu übersetzen und
`0.6` als `NUMERIC`-Score zu schreiben — dann steht im Dashboard eine Zahl, deren
Bedeutung nur im Quelltext nachzulesen ist.

Langfuse löst das über eine **Score-Config**: eine einmalige Projekt-Einstellung,
die zu einem Score-Namen die erlaubten Kategorien samt Zahlenwert festlegt. Ist
sie angelegt, schreibt man weiterhin das Wort — und der Score trägt danach beides:

```
name:        routing-confidence
stringValue: "hoch"        <- das Wort, im Trace direkt lesbar
value:       1.0           <- die Zahl aus der Config, für Mittelwert und Kurve
dataType:    CATEGORICAL
comment:     "Heizung und Warmwasser ausgefallen ..."
```

Die Zahl bleibt dabei eine **Rangkodierung**, keine Messung: 0.3 / 0.6 / 1.0 hält
nur die Reihenfolge fest, die Abstände sind gesetzt. Der Mittelwert ist deshalb
gegen sich selbst über die Zeit vergleichbar, nicht als „so viel Prozent sicher"
lesbar. Der Vorteil der Config ist, dass diese Festlegung an *einer* Stelle steht
— sichtbar im Dashboard statt versteckt in einem Dictionary.

### Wo hängt der Score?

| Aufruf | hängt am | wozu |
| --- | --- | --- |
| `langfuse.score_current_span(...)` | der aktiven Observation, hier `step-routing` | „dieser eine Schritt war unsicher" — sichtbar beim Aufklappen des Traces |
| `langfuse.score_current_trace(...)` | der Trace-Wurzel `tenant-routing-call` | „dieser Anruf war unsicher" — eigene Spalte in der Trace-Liste, aggregierbar |

Beides kommt unten vor: der Span-Score in `route_to_department` (damit er auch
beim direkten Aufruf der Funktion entsteht), der Trace-Score in `handle_call`.

### Ein Haken, den man kennen muss

Mit `config_id` prüft Langfuse den Wert **serverseitig** gegen die Kategorien.
Liefert das Modell ein Wort außerhalb der Liste, wird der Score stillschweigend
verworfen — kein Fehler im Client, nur eine Lücke im Dashboard. Deshalb fängt
`score_confidence()` unten diesen Fall ab und schreibt stattdessen ein
`WARNING`-Event. (Ganz vermeiden lässt er sich, indem man `confidence` im
Pydantic-Modell als `Literal["niedrig", "mittel", "hoch"]` deklariert — siehe
Experimente am Ende.)

Ein Hinweis zur Ehrlichkeit der Zahl: `confidence` ist die Selbsteinschätzung
des Modells, kein gemessener Wert — das Modell benotet sich selbst. Ein Score aus
dieser Quelle sagt „wie sicher war sich das Modell", nicht „wie oft lag es
richtig". Für Letzteres bräuchte es eine zweite Quelle: einen Abgleich gegen
bekannte Soll-Abteilungen (siehe `regression_testing.ipynb`), ein zweites Modell
als Bewerter oder Feedback aus dem Betrieb. Alle drei landen als Score unter
demselben Mechanismus — nur unter einem anderen `name`.


In [ ]:
# Wort -> Zahl. Aus dieser Tabelle entsteht gleich die Score-Config; sie ist damit
# die einzige Stelle, an der die Skala festgelegt wird.
CONFIDENCE_SCORE = {"niedrig": 0.3, "mittel": 0.6, "hoch": 1.0}

# ConfigCategory ist das Paar aus Label und Zahl, aus dem eine Score-Config besteht.
from langfuse.api.commons.types import ConfigCategory


def ensure_confidence_config() -> str | None:
    """Die Score-Config "routing-confidence" holen oder anlegen.

    Eine Score-Config ist eine Projekt-Einstellung, kein Teil eines Traces: Sie
    sagt Langfuse ein fuer alle Mal, welche Kategorien es unter diesem Namen gibt
    und welche Zahl zu welchem Wort gehoert. Ab dann traegt jeder Score beides -
    stringValue "hoch" und value 1.0.

    Idempotent: Erst nachsehen, ob es sie schon gibt. Ohne diese Schleife haette
    man nach dem dritten Ausfuehren der Zelle drei gleichnamige Configs.

    Schlaegt der Aufruf fehl (fehlende Rechte, Netz), gibt die Funktion None
    zurueck. Der Score wird dann trotzdem geschrieben - nur eben ohne die Zahl.
    """
    try:
        for cfg in langfuse.api.score_configs.get(limit=100).data:
            if cfg.name == "routing-confidence" and not cfg.is_archived:
                return cfg.id
        return langfuse.api.score_configs.create(
            name="routing-confidence",
            data_type="CATEGORICAL",
            categories=[ConfigCategory(label=wort, value=zahl)
                        for wort, zahl in CONFIDENCE_SCORE.items()],
            description="Selbsteinschätzung des Modells beim Routing (niedrig/mittel/hoch)",
        ).id
    except Exception as fehler:
        print(f"Score-Config nicht verfügbar ({fehler}) — Scores bekommen nur das Wort.")
        return None


CONFIDENCE_CONFIG_ID = ensure_confidence_config()
print("Score-Config:", CONFIDENCE_CONFIG_ID)


In [ ]:
def score_confidence(scorer, confidence: str, comment: str) -> None:
    """Die Konfidenz als Score schreiben - Wort und Zahl in einem Objekt.

    scorer ist die Methode, die den Score aufhaengt. Der einzige Unterschied
    zwischen den beiden Aufrufstellen:
        langfuse.score_current_span   -> an der aktiven Observation (step-routing)
        langfuse.score_current_trace  -> an der Trace-Wurzel (tenant-routing-call)
    Alles andere ist identisch, deshalb steht es nur hier.
    """
    if confidence not in CONFIDENCE_SCORE:
        # Mit config_id prueft Langfuse serverseitig gegen die Kategorien: Ein Wort
        # ausserhalb der Liste wird stillschweigend verworfen - der Client meldet
        # keinen Fehler, im Dashboard fehlt der Score einfach. Diesen Fall lieber
        # selbst abfangen und sichtbar machen, als eine unerklaerte Luecke zu haben.
        langfuse.create_event(
            name="confidence-unexpected",
            level="WARNING",
            metadata={"confidence": confidence},
            status_message=f"Unerwarteter confidence-Wert: {confidence!r}",
        )
        return

    scorer(
        name="routing-confidence",
        value=confidence,                 # das Wort  -> stringValue
        data_type="CATEGORICAL",          # NUMERIC -> Zahl, CATEGORICAL -> Wort, BOOLEAN -> 0/1
        config_id=CONFIDENCE_CONFIG_ID,   # die Config macht daraus zusaetzlich die Zahl
        comment=comment,                  # Klartext daneben, hier die routing_reason
    )


In [74]:
# Wie in den Schritten davor: ein Span namens "step-routing", darunter haengt der
# OpenAI-Aufruf als Generation. Weil route_to_department() aus handle_call()
# heraus gerufen wird, landet der Span dort als drittes Kind.
@observe(name="step-routing")
def route_to_department(transcript: str) -> RoutingDecision:
    dept_list = "\n".join(f"- {key}: {desc}" for key, desc in DEPARTMENTS.items())
    response = openai.beta.chat.completions.parse(
        model=model,
        messages=[
            {"role": "system", "content": (
                "Du leitest Mieteranfragen bei der Hausverwaltung an die richtige Abteilung weiter. "
                "Wähle genau einen Abteilungs-Schlüssel aus dieser Liste:\n" + dept_list + "\n"
                "Schreibe routing_reason und issue_summary auf Deutsch. "
                "confidence muss genau einer dieser Werte sein: \"niedrig\", \"mittel\", \"hoch\"."
            )},
            {"role": "user", "content": transcript}
        ],
        response_format=RoutingDecision
    )
    decision = response.choices[0].message.parsed

    # score_current_span() haengt den Score an die gerade aktive Observation -
    # das ist innerhalb dieser Funktion der Span "step-routing" selbst. Deshalb
    # steht der Aufruf hier und nicht in handle_call: So entsteht der Score auch
    # dann, wenn route_to_department() direkt aufgerufen wird.
    score_confidence(
        langfuse.score_current_span,   # haengt den Score an "step-routing"
        decision.confidence,
        decision.routing_reason,
    )
    return decision

In [75]:
issue = "Die Heizung ist seit gestern komplett kalt, auch das Warmwasser fehlt."

transcript = f"Eröffnung: {opening}\nName: {name}\nAdresse: {address}\nAnliegen: {issue}"
print(transcript)
print("\n" + "-" * 60 + "\n")

routing = route_to_department(transcript)
print("department:    ", routing.department)
print("routing_reason:", routing.routing_reason)
print("issue_summary: ", routing.issue_summary)
print("confidence:    ", routing.confidence)

Eröffnung: Guten Tag, bei mir in der Wohnung ist die Heizung ausgefallen.
Name: Anna Schmidt
Adresse: Hauptstraße 12, 10115 Berlin
Anliegen: Die Heizung ist seit gestern komplett kalt, auch das Warmwasser fehlt.

------------------------------------------------------------



department:     energy-heating
routing_reason: Die Heizung und Warmwasserversorgung in der Wohnung sind ausgefallen.
issue_summary:  Heizungsausfall und kein Warmwasser in der Wohnung.
confidence:     hoch


## Alles zusammen: ein Trace pro Anruf

Bis hierhin hat jeder Aufruf seinen **eigenen** Trace erzeugt — praktisch zum
Ausprobieren, aber im Dashboard sieht man nicht, was zu welchem Anruf gehörte.
`handle_call` ist wieder mit `@observe` dekoriert und wird damit zum
**Eltern-Trace**, unter dem die drei Schritte als Kinder einsortiert werden.

Der Ablauf im Ganzen: Schritt 1 begrüßt und liefert `reply` + `name`. Ist `name`
leer, fragt der Agent an genau dieser Stelle per `input()` nach — die Zelle hält
also an und wartet auf die Eingabe. Erst mit einem Namen geht es in Schritt 2,
und nur bei `verified=True` weiter in Schritt 3.

`propagate_attributes` reicht Attribute an alles weiter, was innerhalb des
`with`-Blocks passiert:

- **`session_id`** klammert alle Schritte eines Anrufs zusammen. Bei einem
  echten Chatbot würde man hier die Konversations-ID des Nutzers einsetzen,
  dann lassen sich mehrere Anrufe derselben Person im Dashboard gruppieren.
- **`tags`** sind Stichworte zum Filtern — etwa alle abgebrochenen Anrufe
  (`auth-failed`) oder alle mit niedriger Konfidenz.
- **`metadata`** sind freie Schlüssel-Wert-Paare für spätere Auswertungen,
  z. B. "wie oft ging es zu welcher Abteilung?" oder **`name_from_opening`**:
  stand der Name schon in der Eröffnung, oder musste der Agent nachfragen? Im
  Dashboard lässt sich damit auszählen, wie oft die Rückfrage entfallen konnte.

### Events: Zeitpunkte statt Zeiträume

Ein Span misst etwas, das *läuft* — Start, Ende, Dauer, im Dashboard ein Balken.
Ein **Event** markiert dagegen einen Zeitpunkt: „hier ist etwas passiert", ohne
Dauer und ohne Handle, das man später noch aktualisiert. Deshalb ist es auch kein
`with`-Block, sondern der einzelne Aufruf `langfuse.create_event(...)`, der sich
als Kind an den gerade aktiven Span hängt.

Drei solche Meilensteine hält der Anruf fest:

- **`name-requested`** — der Agent musste nach dem Namen fragen.
- **`auth-result`** — das Verifizierungsergebnis mit `verified` und Begründung.
  Fällt es negativ aus, bekommt das Event `level="WARNING"`; die Observation ist
  im Dashboard dann eingefärbt und filterbar.
- **`call-aborted`** bzw. **`department-handoff`** — wie der Anruf ausging.

### Und die Scores am Trace

Ganz am Ende, wenn das Routing feststeht, hängen zwei Scores an der Wurzel:
`routing-confidence` über `score_confidence(...)` — mit Config, also Wort *und*
Zahl — und `routing-department` ohne Config, also nur das Wort. Der Span-Score in
`route_to_department` bleibt davon unberührt: Derselbe Anruf trägt die Konfidenz
zweimal, einmal am Schritt und einmal am ganzen Trace.

Die Ausgabe beginnt mit einer Zeile `[step-greeting] reply=... name=...`: das
rohe `GreetingResult`, damit sichtbar bleibt, was Schritt 1 tatsächlich liefert.
Die Zeile darunter (`Agent: ...`) ist genau das Feld `reply` daraus — also das,
was der Mieter am Telefon hört.

Der Aufruf am Ende ist der eigentliche Zweck des Ganzen: Erst dadurch wird das
Routing-Ergebnis **am Trace** sichtbar und nicht nur in der Konsole.

> Adresse und Anliegen stehen als Parameter da, damit die Zelle sich leicht
> wiederholt ausführen lässt. Im Skript `traceability.py` kommen auch sie über
> `input()` vom Terminal.


In [76]:
# Die aeusserste @observe-Ebene: Dieser Span wird zur Wurzel des Traces, die drei
# Schritt-Spans haengen als Kinder darunter. Unter diesem Namen taucht der Anruf
# in der Trace-Liste des Dashboards auf.
@observe(name="tenant-routing-call")
def handle_call(opening: str, address: str, issue: str):
    session_id = uuid.uuid4().hex[:8]
    # propagate_attributes() tut zweierlei: Es setzt die Attribute sofort auf den
    # gerade aktiven Span - hier die Wurzel, also den Trace selbst - und vererbt
    # sie an jeden Span, der innerhalb des with-Blocks noch entsteht. Deshalb
    # steht der Aufruf so frueh wie moeglich: Rueckwirkend passiert nichts,
    # bereits beendete Spans bleiben ohne die Werte.
    # session_id klammert zusammengehoerige Traces; im Dashboard laesst sich
    # danach gruppieren - bei einem echten Chatbot stuende hier die Konversations-ID.
    with propagate_attributes(session_id=session_id):
        greeting = greet_and_collect_name(opening)
        # Erst das rohe GreetingResult aus Schritt 1, danach die eine Zeile
        # daraus, die der Mieter tatsaechlich hoert.
        print(f"[step-greeting] {greeting}")
        print(f"Agent: {greeting.reply}")

        # Nur wenn Schritt 1 keinen Namen aus der Eroeffnung ziehen konnte,
        # fragt der Agent hier wirklich nach. Die Zelle wartet dann auf die Eingabe.
        if greeting.name:
            name = greeting.name
            herkunft = "aus der Eröffnung"
        else:
            # create_event() legt eine Observation vom Typ "event" an: ein
            # Zeitpunkt ohne Dauer, der sich als Kind an den aktiven Span haengt.
            # Genau richtig fuer Meilensteine wie diesen - "der Agent musste
            # nachfragen" -, wo es nichts zu messen, aber etwas zu vermerken gibt.
            langfuse.create_event(name="name-requested")
            name = input("Mieter (Name): ").strip()
            herkunft = "auf Nachfrage genannt"

        # metadata auf der Trace-Wurzel: haelt fest, ob der Name schon in der
        # Eroeffnung stand. Damit laesst sich spaeter auszaehlen, wie oft die
        # Rueckfrage noetig war.
        with propagate_attributes(metadata={"name_from_opening": greeting.name is not None}):
            print(f"Mieter: {name}   ({herkunft})")

        auth = verify_tenant(name, address)
        # Das Verifizierungsergebnis als Zeitpunkt festhalten - unabhaengig davon,
        # wie es ausfaellt. level faerbt die Observation im Dashboard ein und ist
        # dort filterbar ("DEBUG", "DEFAULT", "WARNING", "ERROR"), status_message
        # nimmt den Klartext dazu.
        langfuse.create_event(
            name="auth-result",
            metadata={"verified": auth.verified, "reason": auth.reason},
            level="DEFAULT" if auth.verified else "WARNING",
            status_message=auth.reason,
        )

        if not auth.verified:
            # tags sind Stichworte am Trace. Ein Filter auf "auth-failed" zeigt
            # im Dashboard genau die abgebrochenen Anrufe.
            with propagate_attributes(tags=["auth-failed"]):
                # Der Abbruch selbst: ein Zeitpunkt, kein Zeitraum. Im Trace-Baum
                # sieht man dadurch, wo der Anruf endete, statt nur zu bemerken,
                # dass danach nichts mehr kommt.
                langfuse.create_event(name="call-aborted", metadata={"stage": "auth"})
                print("Agent: Es tut mir leid, ich konnte Ihre Angaben nicht verifizieren.")
            return None

        print(f"\nAgent: Vielen Dank, {auth.customer_name}. Wie kann ich Ihnen heute helfen?")
        print(f"Mieter: {issue}")

        transcript = f"Eröffnung: {opening}\nName: {name}\nAdresse: {address}\nAnliegen: {issue}"
        routing = route_to_department(transcript)

        # Erst jetzt stehen die Routing-Werte fest, deshalb kommt dieser Block ans
        # Ende. Er setzt tags und metadata auf die Trace-Wurzel; die drei
        # Schritt-Spans sind laengst beendet und bekommen sie nicht mehr ab - fuers
        # Filtern und Auswerten von Traces reicht das.
        with propagate_attributes(
            tags=[routing.department, f"confidence-{routing.confidence}"],
            metadata={
                "routing_department": routing.department,
                "routing_reason": routing.routing_reason,
                "confidence": routing.confidence,
                "customer_name": auth.customer_name,
            }
        ):
            # Derselbe Score noch einmal - aber am Trace statt am Schritt-Span.
            # score_current_trace() haengt ihn an die Wurzel (tenant-routing-call);
            # erst dadurch bekommt der Anruf in der Trace-Liste eine Spalte
            # "routing-confidence", nach der sich sortieren und ueber alle Anrufe
            # hinweg mitteln laesst.
            score_confidence(
                langfuse.score_current_trace,
                routing.confidence,
                f"{routing.department}: {routing.routing_reason}",
            )
            # Zweiter Score, ebenfalls kategorial, aber ohne Config: die gewaehlte
            # Abteilung. Bei Abteilungen gibt es keine sinnvolle Reihenfolge, also
            # auch keine Zahl - der Score traegt nur das Wort (value bleibt 0 und
            # bedeutet nichts). Als Tag ist die Abteilung schon da; der Unterschied:
            # nach einem Tag filtert man, ein kategorialer Score wird im Dashboard
            # als Verteilung ausgezaehlt ("wohin geht wie viel?").
            langfuse.score_current_trace(
                name="routing-department",
                value=routing.department,
                data_type="CATEGORICAL",
            )
            # Der Meilenstein "Ticket ist raus": In einem echten System wuerde
            # hier das Ticketsystem angestossen, im Trace bleibt der Zeitpunkt.
            langfuse.create_event(
                name="department-handoff",
                metadata={"department": routing.department, "confidence": routing.confidence},
            )
            print(f"\n✅ Weiterleitung an: {DEPARTMENTS[routing.department]}")
            print(f"   Grund: {routing.routing_reason}")
            print(f"   Konfidenz: {routing.confidence}")
        return routing


In [77]:
# Eroeffnung MIT Namen: Schritt 1 zieht ihn heraus, es wird nicht nachgefragt.
routing = handle_call(
    opening="Guten Tag, hier ist Anna Schmidt, bei mir in der Wohnung ist die Heizung ausgefallen.",
    address="Hauptstraße 12, 10115 Berlin",
    issue="Die Heizung ist seit gestern komplett kalt, auch das Warmwasser fehlt.",
)


[step-greeting] reply='Guten Tag Frau Schmidt, herzlich willkommen! Es tut mir leid zu hören, dass die Heizung in Ihrer Wohnung ausgefallen ist. Ich werde Ihr Anliegen gerne weiterleiten. Wie kann ich Ihnen weiterhelfen?' name='Anna Schmidt'
Agent: Guten Tag Frau Schmidt, herzlich willkommen! Es tut mir leid zu hören, dass die Heizung in Ihrer Wohnung ausgefallen ist. Ich werde Ihr Anliegen gerne weiterleiten. Wie kann ich Ihnen weiterhelfen?
Mieter: Anna Schmidt   (aus der Eröffnung)

Agent: Vielen Dank, Anna Schmidt. Wie kann ich Ihnen heute helfen?
Mieter: Die Heizung ist seit gestern komplett kalt, auch das Warmwasser fehlt.

✅ Weiterleitung an: Energie & Heizung — Heizungsausfälle, Warmwasser, Nebenkostenabrechnung
   Grund: Heizungsausfall und kein Warmwasser in der Wohnung
   Konfidenz: hoch


In [78]:
# Eroeffnung OHNE Namen: greeting.name ist None -> der Agent fragt per input()
# nach, die Zelle wartet auf die Eingabe (z. B. "Max Mustermann").
# Ausserdem der Abbruch-Pfad: unbrauchbare Adresse -> verified=False -> "auth-failed"
handle_call(
    opening="Hallo, ich habe eine Frage zu meinem Mietvertrag.",
    address="xyz",
    issue="Ich möchte den Vertrag um zwei Jahre verlängern.",
)


[step-greeting] reply='Guten Tag! Wie schön, dass Sie sich melden. Wie darf ich Sie denn nennen?' name=None
Agent: Guten Tag! Wie schön, dass Sie sich melden. Wie darf ich Sie denn nennen?
Mieter: Johannes Riesterer   (auf Nachfrage genannt)
Agent: Es tut mir leid, ich konnte Ihre Angaben nicht verifizieren.


## Traces absenden

Langfuse sammelt die Daten im Hintergrund und schickt sie gebündelt los. Ein
Skript erledigt das beim Beenden — ein Notebook-Kernel läuft aber weiter, deshalb
muss man hier selbst `flush()` aufrufen. Sonst wartet man vergeblich auf Traces
im Dashboard.

Faustregel: nach jedem Experiment ausführen, das man sich ansehen möchte.

In [79]:
# flush() schickt alles, was noch im Puffer liegt, sofort an die Langfuse-API und
# wartet, bis es draussen ist. Normalerweise erledigt das ein Hintergrund-Thread
# in Intervallen bzw. beim Programmende - ein Notebook-Kernel endet aber nicht,
# deshalb hier von Hand.
langfuse.flush()
print("Traces gesendet — jetzt im Langfuse-Dashboard sichtbar.")


Traces gesendet — jetzt im Langfuse-Dashboard sichtbar.


## Experimente

- Ein Anliegen formulieren, das **zwischen zwei Abteilungen** liegt (z. B. "mein
  Nachbar heizt nicht und dadurch schimmelt meine Wand") — welche Abteilung wird
  gewählt, und sinkt die `confidence`?
- Eine Abteilung aus `DEPARTMENTS` entfernen und Schritt 3 neu ausführen:
  Der Prompt ändert sich automatisch mit.
- `confidence` ist bisher nur ein `str` — als
  `Literal["niedrig", "mittel", "hoch"]` deklarieren und beobachten, dass die
  API den Wert dann erzwingt statt ihn nur zu erbitten. Für den Score ist das
  mehr als Kosmetik: Nur so kann das Modell kein Wort liefern, das die
  Score-Config verwirft.
- Eine Kategorie in `CONFIDENCE_SCORE` umbenennen, die Config-Zelle neu ausführen
  (sie findet die alte Config und legt keine neue an) und einen Anruf fahren:
  Der Score fehlt im Dashboard, stattdessen steht dort das `confidence-unexpected`-Event.
- In `GreetingResult` das `| None` bei `name` entfernen und eine Eröffnung ohne
  Namen schicken: Das Modell muss das Feld jetzt füllen — und erfindet einen
  Namen. Ein schönes Beispiel dafür, dass ein Schema Halluzinationen erzwingen
  kann, wenn es keinen Ausweg lässt.
- Zwei Anrufe fahren, einen mit und einen ohne Namen in der Eröffnung, und im
  Dashboard nach `name_from_opening` filtern.
- Mehrere Anrufe mit derselben `session_id` durchführen (Parameter in
  `handle_call` durchreichen) und im Dashboard nach der Session filtern.
- Mehrere Anrufe mit unterschiedlich klaren Anliegen fahren und im Dashboard die
  Zeitreihe von `routing-confidence` ansehen: Sinkt der Durchschnitt, wenn die
  Anliegen unschärfer werden?
- Eine zweite Skala ergänzen, die nicht vom Modell selbst kommt — z. B. nach dem
  Anruf `input("Richtig geroutet? j/n")` abfragen und als
  `data_type="BOOLEAN"`-Score `routing-correct` schreiben. Im Dashboard lässt
  sich dann beides nebeneinander legen: Selbsteinschätzung vs. Wirklichkeit.
- Im Dashboard einen Trace öffnen und die Verschachtelung ansehen:
  `tenant-routing-call` → `step-auth` → `address-format-check`.
